# Treino do detector de estruturas cirúrgicas (YOLOv8, GPU)

Este notebook treina, do zero até pesos prontos, um detector de objetos que reconhece
estruturas anatômicas e instrumentos em imagens de cirurgia laparoscópica. Ele é
**autossuficiente**: não depende de nenhum outro código deste repositório além do
arquivo `finetune.py` que está nesta mesma pasta — só usa a biblioteca `ultralytics`
para treinar e avaliar. O sistema em produção só carrega o peso final (`best.pt`)
para fazer inferência; ele nunca treina.

Treina **dois tamanhos** de modelo base (YOLOv8n e YOLOv8s) sobre o mesmo dataset e
compara os dois contra o split de teste — o vencedor (maior mAP50-95, a métrica mais
rigorosa) é o modelo publicado. YOLOv8n é menor/mais rápido; YOLOv8s tem mais
capacidade e tende a generalizar melhor com mais dados, ao custo de treino/inferência
mais lentos — a comparação real decide, não uma suposição.

**Antes de abrir este notebook**: rode localmente
`python3 training/prepare_dataset_subset.py`, zipe `training/staging/` e suba o zip
para o seu Google Drive (ver `training/README.md` para o passo a passo completo).

**Preparar o ambiente**: no menu do Colab, Ambiente de execução -> Alterar tipo de
ambiente de execução -> GPU.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste para o caminho onde você subiu o zip preparado no passo anterior.
DRIVE_STAGING_ZIP = '/content/drive/MyDrive/endoscapes_staging.zip'
# Tudo que este notebook produz (pesos, métricas) é salvo direto no Drive -- a
# sessão do Colab é temporária e pode cair por inatividade a qualquer momento.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/endoscapes_training_output'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!pip install -q ultralytics

# Suba também o arquivo `finetune.py` (desta mesma pasta `training/`) para o
# Colab -- use o painel de arquivos à esquerda (ícone de pasta) e arraste o
# arquivo para a raiz de `/content/`, ou rode a célula abaixo se preferir
# colar o conteúdo diretamente aqui.
import sys
sys.path.insert(0, '/content')
from finetune import finetune

In [16]:
import shutil
from pathlib import Path

staging_dir = Path('/content/dataset_staging')
shutil.unpack_archive(DRIVE_STAGING_ZIP, staging_dir)

# O dataset já vem dividido oficialmente em treino/validação/teste -- nunca
# misturar imagens entre esses grupos: imagens vizinhas de um mesmo vídeo
# cirúrgico são quase idênticas, e misturá-las inflaria artificialmente a
# métrica final (o modelo pareceria acertar imagens que na prática já viu).
train_coco = staging_dir / 'staging' / 'train' / 'annotation_coco.json'
train_images = staging_dir / 'staging'/ 'train'
val_coco = staging_dir / 'staging'/ 'val' / 'annotation_coco.json'
val_images = staging_dir / 'staging'/ 'val'
test_coco = staging_dir / 'staging'/ 'test' / 'annotation_coco.json'
test_images = staging_dir / 'staging'/ 'test'

for arquivo in (train_coco, val_coco, test_coco):
    assert arquivo.is_file(), f'{arquivo} ausente -- confira o zip subido ao Drive'

## Treino

Usa o split de validação de verdade durante o treino (não o próprio treino outra
vez) e guarda o resultado direto no Drive. Treina cada variante em sequência (a GPU
grátis do Colab só roda um treino por vez) — YOLOv8n primeiro, depois YOLOv8s.

In [17]:
SEED = 42
EPOCHS = 100
IMGSZ = 640
MODEL_VARIANTS = ['yolov8n', 'yolov8s']

best_weights_by_variant = {}
for variant in MODEL_VARIANTS:
    print(f'--- treinando {variant} ---')
    best_weights_by_variant[variant] = finetune(
        train_coco=train_coco,
        train_images=train_images,
        val_coco=val_coco,
        val_images=val_images,
        output_dir=Path(DRIVE_OUTPUT_DIR),
        model_variant=variant,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        seed=SEED,
    )
    print(f'{variant} treinado em:', best_weights_by_variant[variant])

--- treinando yolov8n ---
Annotations /content/drive/MyDrive/endoscapes_training_output/yolo/_coco_src_train/annotation_coco.json: 100% ━━━━━━━━━━━━ 1186/1186 68.1it/s 17.4s
COCO data converted successfully.
Results saved to /content/drive/MyDrive/endoscapes_training_output/yolo/_tmp_train
Annotations /content/drive/MyDrive/endoscapes_training_output/yolo/_coco_src_val/annotation_coco.json: 100% ━━━━━━━━━━━━ 367/367 93.7it/s 3.9s
COCO data converted successfully.
Results saved to /content/drive/MyDrive/endoscapes_training_output/yolo/_tmp_val
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/endoscapes_training_output/yolo/

In [18]:
# Guarda o histórico de métricas por época de cada variante ao lado do peso final,
# mais fácil de achar depois.
for variant, best_weights in best_weights_by_variant.items():
    results_csv = best_weights.parent.parent / 'results.csv'
    if results_csv.is_file():
        dest = Path(DRIVE_OUTPUT_DIR) / f'results_{variant}.csv'
        shutil.copy2(results_csv, dest)
        print(f'Histórico de treino de {variant} salvo em', dest)

Histórico de treino de yolov8n salvo em /content/drive/MyDrive/endoscapes_training_output/results_yolov8n.csv
Histórico de treino de yolov8s salvo em /content/drive/MyDrive/endoscapes_training_output/results_yolov8s.csv


## Avaliação honesta contra o conjunto de teste (nunca visto no treino) e escolha do vencedor

Usa a própria ferramenta de validação do `ultralytics` sobre o split de teste, para
cada variante treinada — essa é a métrica que importa para o relatório, não o
desempenho durante o treino. O vencedor entre YOLOv8n e YOLOv8s é escolhido pelo
mAP50-95 (mAP médio sobre vários limiares de IoU — mais rigoroso que o mAP50 porque
também pune caixas mal localizadas, não só classificação errada).

In [19]:
import json

from finetune import convert_split, coco_categories
from ultralytics import YOLO

yolo_dir = Path(DRIVE_OUTPUT_DIR) / 'yolo'
convert_split(test_coco, test_images, yolo_dir, 'test')

# O ultralytics exige as chaves `train`/`val` em qualquer YAML de dataset,
# mesmo quando só queremos validar -- apontamos as duas para o próprio split
# de teste; só a métrica de `split='val'` abaixo é usada de fato.
nomes_yaml = ''.join(
    f'  {idx}: {nome}\n' for idx, nome in sorted(coco_categories(test_coco).items())
)
test_dataset_yaml = yolo_dir / 'dataset_test.yaml'
test_dataset_yaml.write_text(
    f'path: {yolo_dir}\ntrain: images/test\nval: images/test\nnames:\n{nomes_yaml}',
    encoding='utf-8',
)

resumo_por_variante = {}
for variant, weights_path in best_weights_by_variant.items():
    modelo = YOLO(str(weights_path))
    # `project=`/`name=` explícitos evitam que o ultralytics grave em `runs/`
    # relativo ao diretório de trabalho corrente (mesmo cuidado do treino acima);
    # `name` por variante evita que uma avaliação sobrescreva a da outra.
    metricas = modelo.val(
        data=str(test_dataset_yaml),
        split='val',
        project=str(Path(DRIVE_OUTPUT_DIR) / 'runs'),
        name=f'avaliacao_teste_{variant}',
        exist_ok=True,
    )
    resumo_por_variante[variant] = {
        'precision_media': float(metricas.box.mp),
        'recall_medio': float(metricas.box.mr),
        'mAP50': float(metricas.box.map50),
        'mAP50_95': float(metricas.box.map),
    }
    print(variant, resumo_por_variante[variant])

melhor_variante = max(resumo_por_variante, key=lambda v: resumo_por_variante[v]['mAP50_95'])
best_weights = best_weights_by_variant[melhor_variante]

metrics_path = Path(DRIVE_OUTPUT_DIR) / 'metrics_test.json'
metrics_path.write_text(
    json.dumps(
        {'por_variante': resumo_por_variante, 'melhor_variante': melhor_variante},
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(f'\nMelhor modelo: {melhor_variante} (mAP50-95={resumo_por_variante[melhor_variante]["mAP50_95"]:.4f})')
print('Pesos vencedores em:', best_weights)
print('Métricas de teste salvas em', metrics_path)

Annotations /content/drive/MyDrive/endoscapes_training_output/yolo/_coco_src_test/annotation_coco.json: 100% ━━━━━━━━━━━━ 289/289 95.5it/s 3.0s
COCO data converted successfully.
Results saved to /content/drive/MyDrive/endoscapes_training_output/yolo/_tmp_test
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.4±0.4 ms, read: 33.9±20.5 MB/s, size: 113.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/endoscapes_training_output/yolo/labels/test... 289 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 312/312 121.3it/s 2.6s
val: New cache created: /content/drive/MyDrive/endoscapes_training_output/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50

## Próximos passos (fora do Colab)

1. Confira em `metrics_test.json` (`DRIVE_OUTPUT_DIR`) o campo `melhor_variante` —
   diz qual dos dois (`yolov8n` ou `yolov8s`) venceu. Baixe **só o `best.pt` dessa
   variante** (em `DRIVE_OUTPUT_DIR/runs/finetune_<melhor_variante>/weights/best.pt`),
   junto com `results_<melhor_variante>.csv` e `metrics_test.json`, para a pasta
   `models/` deste repositório, na sua máquina.
2. Preencha `models/README.md` com data, hiperparâmetros usados, qual variante venceu
   e as métricas de `metrics_test.json`.
3. Publique `best.pt` no repositório de modelo público do
   [Hugging Face Hub](https://huggingface.co/AnaPRodrigues/endoscapes-surgical-detector)
   (pela interface web, aba "Files and versions" → "Add file" → "Upload files", ou via
   `huggingface-cli upload AnaPRodrigues/endoscapes-surgical-detector best.pt`).
4. Quem só quer rodar o sistema (sem retreinar) executa `make models-fetch`, que
   baixa esse peso publicado automaticamente.